# Fabrication Analysis — Text Extraction & V1 vs V2 Validation

**Pipeline overview:**
1. **Phase 1** — OCR text extraction for 14 confirmed fabrication cases → `fabrication_analysis/extracted_text/`
2. **Phase 2** — Build ground truth table from `merged_llm_summary_validation_datasheet.xlsx` + `ai_fabrications_dataset.xlsx`
3. **Phase 3** — Load v1 (narrative DOCX) and v2 (structured JSON TXT) summaries
4. **Phase 4** — LLM-based validation of v2 summaries against source text (GPT-4o)
5. **Phase 5** — Audit table + comparison figures: which v1 fabrications were corrected in v2?

In [ ]:
import os
import sys
import json
import re
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from docx import Document
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

PROJECT_ROOT = Path(
    os.getenv('PROJECT_ROOT',
    r'C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center'
    r'\Documents\GitHub\llm_summarization_br_ca')
)
DATA = Path(r'C:\Users\jamesr4\loc\data_private')
sys.path.insert(0, str(PROJECT_ROOT))

# Directories
OCR_CACHE       = DATA / 'ocr_cache'
MAPPING_CSV     = DATA / 'breast_bot_deidentified' / 'case_id_mapping.csv'
FAB_XLSX        = DATA / 'raw' / 'ai_fabrications_dataset.xlsx'
GT_XLSX         = Path(
    r'C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center'
    r'\Documents\Research\Projects\moo\llm_summary\data\raw'
    r'\merged_llm_summary_validation_datasheet.xlsx'
)
V1_PATHS_CSV    = DATA / 'raw' / 'v1_summary_paths.csv'
V2_PATHS_CSV    = DATA / 'raw' / 'v2_summary_paths.csv'

FAB_ANALYSIS_DIR = PROJECT_ROOT / 'fabrication_analysis'
EXTRACTED_TEXT   = FAB_ANALYSIS_DIR / 'extracted_text'
CACHE_DIR        = FAB_ANALYSIS_DIR / 'cache'
REPORTS_DIR      = PROJECT_ROOT / 'reports'

for d in [EXTRACTED_TEXT, CACHE_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'EXTRACTED_TEXT: {EXTRACTED_TEXT}')
print(f'GT_XLSX exists: {GT_XLSX.exists()}')
print(f'FAB_XLSX exists: {FAB_XLSX.exists()}')

## Phase 1 — OCR Text Extraction for 14 Fabrication Cases
Uses pre-cached OCR files (`ocr_cache/*.txt`) assembled per-case following the same
document-concatenation approach as `notebooks/04_executed.ipynb`. Writes one `.txt`
per case to `fabrication_analysis/extracted_text/`.

In [ ]:
_SURGEON_MAP = {
    'el tamer': 'el tamer', 'el-tamer': 'el tamer',
    'sacchini': 'sacchini', 'giannakou': 'giannakou',
    'montagna': 'montag', 'montag': 'montag', 'lisa allen': 'allen',
}
def _norm(s):
    return _SURGEON_MAP.get(str(s).strip().lower(), str(s).strip().lower())

# Load fab cases
fab = pd.read_excel(FAB_XLSX)
ai_cols = [c for c in fab.columns if c.endswith('_status_ai')]
for c in ai_cols:
    fab[c] = pd.to_numeric(fab[c], errors='coerce')
fab['surgeon_last'] = fab['surgeon'].str.split(',').str[0].str.strip()
fab['surgeon_norm'] = fab['surgeon_last'].apply(_norm)
fab['fab_features'] = fab.apply(
    lambda r: [c.replace('_status_ai', '') for c in ai_cols if r[c] == 3], axis=1
)

# Build case_folder mapping
mapping = pd.read_csv(MAPPING_CSV)
mapping['surgeon_norm']    = mapping['surgeon'].apply(_norm)
mapping['folder_initials'] = mapping['case_folder'].str.split('_').str[1].str.upper()
rep = (
    mapping.dropna(subset=['original_path'])
    .drop_duplicates(subset=['surgeon_norm', 'folder_initials'])
    [['surgeon_norm', 'folder_initials', 'case_folder']]
)
df = fab.merge(rep, left_on=['surgeon_norm', 'patient_initials'],
               right_on=['surgeon_norm', 'folder_initials'], how='left')

# Extract and write per-case text files
extraction_log = []
fab_cases = []

for _, row in df.iterrows():
    cf      = str(row.get('case_folder', ''))
    mrn     = int(row['mrn'])
    doc_ids = mapping[mapping['case_folder'] == cf][['case_id', 'original_filename']].copy()

    texts = []
    for _, doc_row in doc_ids.iterrows():
        cache_file = OCR_CACHE / f"{doc_row['case_id']}.txt"
        if cache_file.exists():
            content = cache_file.read_text(encoding='utf-8', errors='ignore')
            texts.append(f"\n[DOCUMENT: {doc_row['original_filename']}]\n{content}")

    full_text  = '\n\n--- PAGE BREAK ---\n\n'.join(texts)
    out_path   = EXTRACTED_TEXT / f'{cf}.txt'
    out_path.write_text(full_text, encoding='utf-8')

    extraction_log.append({
        'mrn': mrn, 'case_folder': cf,
        'n_docs': len(doc_ids), 'n_cached': len(texts),
        'char_count': len(full_text), 'output_file': str(out_path),
    })
    fab_cases.append({
        'mrn': mrn, 'case_folder': cf,
        'fab_features': row['fab_features'],
        'ocr_text': full_text,
    })
    print(f'  {cf:<22} {len(texts)} docs → {len(full_text):>7} chars  fab={row["fab_features"]}')

log_df = pd.DataFrame(extraction_log)
log_df.to_csv(FAB_ANALYSIS_DIR / 'extraction_log.csv', index=False)

print(f'\nExtracted {len(fab_cases)} cases → {EXTRACTED_TEXT}')
print(f'Extraction log: {FAB_ANALYSIS_DIR / "extraction_log.csv"}')
log_df

## Phase 2 — Ground Truth from Validation Datasheet
Loads `merged_llm_summary_validation_datasheet.xlsx` (Validation sheet) to get:
- Per-case feature status codes (`_status_source`, `_status_ai`)
- Reviewer comments documenting the specific v1 fabrication issue
- Confirmed v1 fabrications from `ai_fabrications_dataset.xlsx` (status_ai == 3)

In [ ]:
# Load validation datasheet
val_df = pd.read_excel(GT_XLSX, sheet_name='Validation')
val_df['mrn'] = pd.to_numeric(val_df['mrn'], errors='coerce')
val_df = val_df.dropna(subset=['mrn'])
val_df['mrn'] = val_df['mrn'].astype(int)

# Feature status columns
FEATURE_COLS = {
    'lesion_size':                        ('lesion_size_status_source',                        'lesion_size_status_ai'),
    'laterality':                         ('laterality_status_source',                         'laterality_status_ai'),
    'lesion_location':                    ('lesion_location_status_source',                    'lesion_location_status_ai'),
    'calcifications_asymmetry':           ('calcifications_asymmetry_status_source',           'calcifications_asymmetry_status_ai'),
    'additional_enhancement_mri':         ('additional_enhancement_mri_status_source',         'additional_enhancement_mri_status_ai'),
    'extent':                             ('extent_status_source',                             'extent_status_ai'),
    'accurate_clip_placement':            ('accurate_clip_placement_status_source',            'accurate_clip_placement_status_ai'),
    'workup_recommendation':              ('workup_recommendation_status_source',              'workup_recommendation_status_ai'),
    'lymph_node':                         ('Lymph node_status_source',                        'Lymph node_status_ai'),
    'chronology_preserved':               ('chronology_preserved_status_source',               'chronology_preserved_status_ai'),
    'biopsy_method':                      ('biopsy_method_status_source',                     'biopsy_method_status_ai'),
    'invasive_component_size_pathology':  ('invasive_component_size_pathology_status_source',  'invasive_component_size_pathology_status_ai'),
    'histologic_diagnosis':               ('histologic_diagnosis_status_source',               'histologic_diagnosis_status_ai'),
    'receptor':                           ('receptor_status_source',                           'receptor_status_ai'),
}

# Build ground truth rows for the 14 fab cases
fab_mrns = [c['mrn'] for c in fab_cases]
gt_rows  = []

for case in fab_cases:
    mrn      = case['mrn']
    cf       = case['case_folder']
    val_rows = val_df[val_df['mrn'] == mrn]
    in_sheet = not val_rows.empty

    for feat in case['fab_features']:
        cols = FEATURE_COLS.get(feat)
        if in_sheet and cols:
            source_col, ai_col = cols
            src_val = val_rows[source_col].values[0] if source_col in val_rows.columns else None
            ai_val  = val_rows[ai_col].values[0]    if ai_col  in val_rows.columns else None
            comment = val_rows['comments'].values[0] if 'comments' in val_rows.columns else ''
        else:
            src_val, ai_val, comment = None, None, ''

        gt_rows.append({
            'mrn':              mrn,
            'case_folder':      cf,
            'fab_feature':      feat,
            'in_validation_sheet': in_sheet,
            'source_status':    src_val,
            'v1_ai_status':     ai_val,
            'reviewer_comment': str(comment) if pd.notna(comment) else '',
        })

gt_df = pd.DataFrame(gt_rows)
gt_df.to_csv(FAB_ANALYSIS_DIR / 'ground_truth_table.csv', index=False)

print(f'Ground truth rows: {len(gt_df)}')
print(f'Cases in validation sheet: {gt_df["in_validation_sheet"].sum()}/{len(gt_df)}')
print()
gt_df

## Phase 3 — Load V1 and V2 Summaries

In [ ]:
v1_paths = pd.read_csv(V1_PATHS_CSV)
v2_paths = pd.read_csv(V2_PATHS_CSV)

def read_docx_text(path_str):
    """Extract full text from a .docx file."""
    p = Path(path_str)
    if not p.exists():
        return f'[FILE NOT FOUND: {path_str}]'
    try:
        doc = Document(str(p))
        return '\n'.join(para.text for para in doc.paragraphs if para.text.strip())
    except Exception as e:
        return f'[ERROR reading DOCX: {e}]'

def read_v2_json(path_str):
    """Read v2 structured JSON txt summary and return as dict."""
    p = Path(path_str)
    if not p.exists():
        return {}
    raw = p.read_text(encoding='utf-8', errors='ignore')
    # Strip markdown code blocks if present
    raw = re.sub(r'^```[a-z]*\n?', '', raw.strip(), flags=re.MULTILINE)
    raw = re.sub(r'```$', '', raw.strip())
    try:
        return json.loads(raw)
    except Exception:
        return {'_raw': raw}

def v2_feature_value(parsed_json, feature_name):
    """Retrieve {value, evidence} for a feature from v2 JSON."""
    key_variants = [
        feature_name,
        feature_name.replace('_', ' '),
        feature_name.replace('_status', ''),
    ]
    for key in parsed_json:
        if any(v.lower() in key.lower() or key.lower() in v.lower()
               for v in key_variants):
            val = parsed_json[key]
            if isinstance(val, dict):
                return val
            return {'value': str(val), 'evidence': ''}
    return None

# Build summary lookup: mrn → {v1_text, v2_json}
summaries = {}
for case in fab_cases:
    mrn = case['mrn']
    v1_row = v1_paths[v1_paths['mrn'] == mrn]
    v2_row = v2_paths[v2_paths['mrn'] == mrn]
    v1_text = read_docx_text(v1_row['summary_path'].values[0]) if len(v1_row) else '[V1 NOT FOUND]'
    v2_data = read_v2_json(v2_row['summary_path'].values[0])   if len(v2_row) else {}
    summaries[mrn] = {'v1_text': v1_text, 'v2_json': v2_data,
                      'case_folder': case['case_folder']}

print(f'Loaded summaries for {len(summaries)} cases')
for mrn, s in summaries.items():
    v1_ok = not s['v1_text'].startswith('[')  
    v2_ok = bool(s['v2_json']) and '_raw' not in s['v2_json']
    print(f'  MRN {mrn}  v1={'OK' if v1_ok else 'MISSING'}  v2={"OK" if v2_ok else "RAW/MISSING"}')

## Phase 4 — LLM-Based Validation of V2 Against Source Text
For each confirmed v1 fabrication, asks GPT-4o:
- What does the SOURCE document actually say about this feature?
- Does the V2 summary correctly report it (corrected) or still fabricate?

Results cached to `fabrication_analysis/cache/v2_validation_cache.json`.

In [ ]:
CACHE_FILE = CACHE_DIR / 'v2_validation_cache.json'
cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

SYSTEM_PROMPT = """You are a clinical data quality reviewer specializing in breast oncology.
Your task: determine if an AI-generated summary correctly reports a specific clinical feature
based on what the source documents actually state.
Respond ONLY with valid JSON. No other text."""

FEATURE_DESCRIPTIONS = {
    'lesion_size':                       'Primary lesion size (cm or mm) from imaging or pathology',
    'laterality':                        'Breast side — left or right',
    'lesion_location':                   'Quadrant and/or clock-face position of the lesion',
    'calcifications_asymmetry':          'Presence and description of calcifications or asymmetry',
    'additional_enhancement_mri':        'Additional MRI enhancement beyond the index lesion',
    'extent':                            'Extent of disease on MRI (cm)',
    'accurate_clip_placement':           'Whether biopsy clip was placed accurately',
    'workup_recommendation':             'Recommended next workup steps',
    'lymph_node':                        'Axillary lymph node status',
    'chronology_preserved':              'Chronological ordering of events is correct',
    'biopsy_method':                     'Method used for biopsy (e.g., US-guided core needle)',
    'invasive_component_size_pathology': 'Invasive tumor component size on final pathology (cm or mm)',
    'histologic_diagnosis':              'Histologic type of carcinoma (e.g., IDC, ILC, DCIS)',
    'receptor':                          'Hormone receptor and HER2 status (ER, PR, HER2)',
}

def validate_v2_vs_source(mrn, case_folder, feature, ocr_text,
                           v2_json, reviewer_comment, v1_text):
    """Ask GPT-4o whether v2 correctly reports a fabricated feature vs source."""
    cache_key = f'{mrn}_{feature}'
    if cache_key in cache:
        return cache[cache_key]

    feat_desc  = FEATURE_DESCRIPTIONS.get(feature, feature.replace('_', ' '))
    v2_feat    = v2_feature_value(v2_json, feature)
    v2_val     = v2_feat.get('value', 'NOT FOUND')     if v2_feat else 'NOT FOUND IN V2'
    v2_evid    = v2_feat.get('evidence', '')[:600]     if v2_feat else ''

    user_msg = f"""CONFIRMED V1 FABRICATION CASE
MRN: {mrn}  |  Case: {case_folder}  |  Feature: {feature.upper()}
Feature description: {feat_desc}

REVIEWER NOTE about V1 fabrication:
{reviewer_comment if reviewer_comment else '(no specific comment — status_ai=3 in dataset)'}

SOURCE DOCUMENT TEXT (first 3000 chars):
---
{ocr_text[:3000]}
---

V1 SUMMARY EXCERPT (narrative, known to contain fabrication):
---
{v1_text[:1500]}
---

V2 SUMMARY EXTRACTED VALUE (structured JSON):
  value    : {v2_val}
  evidence : {v2_evid}

TASK:
1. Based on the source text, what is the CORRECT value for '{feature}'? Quote the source.
2. Is the V2 extracted value consistent with the source document?
3. Was the V1 fabrication CORRECTED in V2, or does V2 still contain an error?

Return JSON:
{{
  "source_ground_truth": "verbatim or close quote from source document",
  "v2_verdict": "CORRECT | FABRICATION | OMISSION | UNCERTAIN",
  "corrected": true_or_false,
  "confidence": 0.0_to_1.0,
  "explanation": "brief explanation"
}}"""

    try:
        resp = client.chat.completions.create(
            model='gpt-4o',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=0.0,
            max_tokens=512,
            response_format={'type': 'json_object'},
        )
        raw = resp.choices[0].message.content
        result = json.loads(raw)
        result['mrn']     = mrn
        result['feature'] = feature
    except Exception as e:
        result = {
            'mrn': mrn, 'feature': feature,
            'v2_verdict': 'ERROR', 'corrected': None,
            'confidence': 0.0, 'explanation': str(e),
            'source_ground_truth': '',
        }

    cache[cache_key] = result
    return result

# Run validation for all 14 × n_features
validation_results = []
case_lookup = {c['mrn']: c for c in fab_cases}

for _, gt_row in gt_df.iterrows():
    mrn     = gt_row['mrn']
    feat    = gt_row['fab_feature']
    comment = gt_row['reviewer_comment']
    case    = case_lookup[mrn]
    sums    = summaries[mrn]

    print(f'  Validating MRN {mrn} | {feat} ...', end=' ', flush=True)
    result = validate_v2_vs_source(
        mrn=mrn,
        case_folder=case['case_folder'],
        feature=feat,
        ocr_text=case['ocr_text'],
        v2_json=sums['v2_json'],
        reviewer_comment=comment,
        v1_text=sums['v1_text'],
    )
    print(f"{result.get('v2_verdict','?')}  corrected={result.get('corrected','?')}")
    time.sleep(0.5)  # rate-limit buffer

    validation_results.append({
        'mrn':              mrn,
        'case_folder':      case['case_folder'],
        'fab_feature':      feat,
        'source_status':    gt_row['source_status'],
        'v1_ai_status':     gt_row['v1_ai_status'],
        'reviewer_comment': comment,
        'source_ground_truth': result.get('source_ground_truth', ''),
        'v2_verdict':       result.get('v2_verdict', 'ERROR'),
        'corrected':        result.get('corrected', None),
        'confidence':       result.get('confidence', 0.0),
        'explanation':      result.get('explanation', ''),
    })

# Save cache
CACHE_FILE.write_text(json.dumps(cache, indent=2))
print(f'\nCache saved → {CACHE_FILE}')

## Phase 5 — Audit Table & Comparison Figures

In [ ]:
# Build audit dataframe
audit_df = pd.DataFrame(validation_results)
audit_df['v1_fabricated'] = True  # all rows are confirmed v1 fabrications
audit_df['corrected_in_v2'] = audit_df['corrected'].fillna(False).astype(bool)
audit_df['v2_verdict'] = audit_df['v2_verdict'].fillna('UNKNOWN')

# Save audit table
audit_path = FAB_ANALYSIS_DIR / 'audit_table_v1_vs_v2.csv'
audit_df.to_csv(audit_path, index=False)

# Summary stats
n_total     = len(audit_df)
n_corrected = audit_df['corrected_in_v2'].sum()
n_persisted = (~audit_df['corrected_in_v2']).sum()
pct_corrected = n_corrected / n_total * 100 if n_total > 0 else 0

print('=' * 60)
print('FABRICATION CORRECTION SUMMARY: V1 → V2')
print('=' * 60)
print(f'  Total confirmed v1 fabrications : {n_total}')
print(f'  Corrected in v2                 : {n_corrected}  ({pct_corrected:.0f}%)')
print(f'  Persisted / still wrong in v2   : {n_persisted}  ({100-pct_corrected:.0f}%)')
print()
print('V2 verdict breakdown:')
print(audit_df['v2_verdict'].value_counts().to_string())
print()
print('Audit table preview:')
print(audit_df[['case_folder','fab_feature','v1_fabricated','v2_verdict','corrected_in_v2',
               'confidence']].to_string(index=False))

In [ ]:
# ── Figure 1: V1 Fabrications Corrected vs Persisted in V2 ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

COLORS = {
    'CORRECT':     '#2ecc71',
    'FABRICATION': '#e74c3c',
    'OMISSION':    '#f39c12',
    'UNCERTAIN':   '#95a5a6',
    'ERROR':       '#bdc3c7',
    'UNKNOWN':     '#bdc3c7',
}

# A: Pie — overall correction rate
ax = axes[0]
sizes  = [n_corrected, n_persisted]
labels = [f'Corrected in V2\n({n_corrected}/{n_total})',
          f'Persisted / new error\n({n_persisted}/{n_total})']
clrs   = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=clrs,
    autopct='%1.0f%%', startangle=90,
    textprops={'fontsize': 10}, pctdistance=0.75,
)
for at in autotexts:
    at.set_fontweight('bold')
ax.set_title('Overall: V1 Fabrications\nCorrected in V2', fontweight='bold', fontsize=12)

# B: Bar — by feature
ax = axes[1]
feat_summary = (
    audit_df.groupby('fab_feature')['corrected_in_v2']
    .agg(['sum', 'count'])
    .reset_index()
    .rename(columns={'sum': 'corrected', 'count': 'total'})
)
feat_summary['pct'] = feat_summary['corrected'] / feat_summary['total'] * 100
feat_summary = feat_summary.sort_values('pct', ascending=True)
bar_colors = ['#2ecc71' if p == 100 else '#f39c12' if p >= 50 else '#e74c3c'
              for p in feat_summary['pct']]
bars = ax.barh(feat_summary['fab_feature'], feat_summary['pct'],
               color=bar_colors, edgecolor='white')
for bar, (_, row) in zip(bars, feat_summary.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{int(row['corrected'])}/{int(row['total'])}",
            va='center', fontsize=9)
ax.set_xlim(0, 120)
ax.axvline(100, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Correction Rate (%)')
ax.set_title('Correction Rate\nby Fabricated Feature', fontweight='bold', fontsize=12)
patches = [
    mpatches.Patch(color='#2ecc71', label='100% corrected'),
    mpatches.Patch(color='#f39c12', label='50-99% corrected'),
    mpatches.Patch(color='#e74c3c', label='<50% corrected'),
]
ax.legend(handles=patches, fontsize=8, loc='lower right')

# C: V2 Verdict breakdown (stacked bar per feature)
ax = axes[2]
verdict_pivot = (
    audit_df.groupby(['fab_feature', 'v2_verdict'])
    .size().unstack(fill_value=0)
)
# Reorder columns
ordered_verdicts = [v for v in ['CORRECT', 'OMISSION', 'UNCERTAIN', 'FABRICATION', 'ERROR']
                    if v in verdict_pivot.columns]
verdict_pivot = verdict_pivot[ordered_verdicts]
verdict_pivot.plot(
    kind='barh', stacked=True, ax=ax,
    color=[COLORS.get(v, '#bdc3c7') for v in ordered_verdicts],
    edgecolor='white',
)
ax.set_title('V2 Verdict Distribution\nby Feature', fontweight='bold', fontsize=12)
ax.set_xlabel('Count')
ax.legend(title='V2 Verdict', fontsize=8)

plt.suptitle(
    'V1 → V2 Fabrication Correction Analysis\n'
    f'14 Confirmed Cases | {n_corrected}/{n_total} ({pct_corrected:.0f}%) Corrected',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
fig1_path = REPORTS_DIR / 'v1_v2_fabrication_correction_summary.png'
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig1_path}')

In [ ]:
# ── Figure 2: Per-Case Correction Heatmap ────────────────────────────────────
heat_data = audit_df.pivot_table(
    index='case_folder', columns='fab_feature',
    values='corrected_in_v2', aggfunc='first',
).fillna(False).astype(float)

# Annotate with v2 verdict
annot_data = audit_df.pivot_table(
    index='case_folder', columns='fab_feature',
    values='v2_verdict', aggfunc='first',
).fillna('')

fig2, ax2 = plt.subplots(figsize=(max(10, len(heat_data.columns) * 1.5),
                                  max(5, len(heat_data) * 0.6)))
sns.heatmap(
    heat_data, ax=ax2,
    cmap=['#e74c3c', '#2ecc71'],
    vmin=0, vmax=1,
    annot=annot_data, fmt='',
    linewidths=0.8, linecolor='white',
    cbar_kws={'ticks': [0, 1], 'label': 'Corrected (1=Yes, 0=No)'},
)
ax2.set_title(
    'Per-Case Fabrication Correction Status: V1 → V2\n'
    '(Green = corrected in V2, Red = persisted/new error)',
    fontsize=13, fontweight='bold'
)
ax2.set_xlabel('Fabricated Feature (confirmed in V1)')
ax2.set_ylabel('Case')
plt.tight_layout()
fig2_path = REPORTS_DIR / 'v1_v2_correction_heatmap.png'
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig2_path}')

In [ ]:
# ── Figure 3: Confidence Distribution + Detailed Audit Text ──────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))

# A: Confidence by verdict
ax = axes3[0]
for verdict, group in audit_df.groupby('v2_verdict'):
    ax.hist(group['confidence'].dropna(), bins=10, alpha=0.65,
            color=COLORS.get(verdict, '#bdc3c7'), label=verdict)
ax.set_title('LLM Judge Confidence\nby V2 Verdict', fontweight='bold')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.axvline(0.75, color='black', linestyle='--', alpha=0.5, label='0.75 threshold')
ax.legend(fontsize=9)

# B: V1 prompt vs V2 prompt side-by-side fabrication rate comparison
ax = axes3[1]
comparison = pd.DataFrame([
    {'prompt': 'V1 (narrative, no constraints)',
     'fabrication_rate': 1.0,
     'n_cases': n_total},
    {'prompt': 'V2 (structured JSON, anti-fab constraints)',
     'fabrication_rate': n_persisted / n_total,
     'n_cases': n_total},
])
bar_c = ['#e74c3c', '#2ecc71' if n_persisted == 0 else '#f39c12' if n_persisted < n_total/2 else '#e74c3c']
bars3 = ax.bar(comparison['prompt'], comparison['fabrication_rate'] * 100,
               color=bar_c, edgecolor='white', width=0.5)
for bar, (_, row) in zip(bars3, comparison.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{row['fabrication_rate']:.0%}",
            ha='center', fontweight='bold', fontsize=12)
ax.set_ylim(0, 120)
ax.set_ylabel('Fabrication Rate (%)')
ax.set_title('Fabrication Rate:\nV1 Prompt vs V2 Prompt\n(Same 14 Cases)', fontweight='bold')
ax.tick_params(axis='x', labelsize=9)

plt.suptitle('Prompt Improvement Analysis: V1 → V2', fontsize=12, fontweight='bold')
plt.tight_layout()
fig3_path = REPORTS_DIR / 'v1_v2_prompt_improvement.png'
plt.savefig(fig3_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig3_path}')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
saved = sorted(REPORTS_DIR.glob('v1_v2*.png'))
print('=== FINAL SUMMARY ===')
print(f'Audit table : {audit_path}')
print(f'Figures     : {[p.name for p in saved]}')
print()
print('DETAILED AUDIT TABLE:')
cols = ['case_folder','fab_feature','v2_verdict','corrected_in_v2',
        'confidence','source_ground_truth','explanation']
pd.set_option('display.max_colwidth', 80)
print(audit_df[cols].to_string(index=False))